<a href="https://colab.research.google.com/github/jordanmsouza/TechChallenge_Fase4_Grupo4/blob/main/TechChallenge_Fase4_Grupo4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Instalando dependências

In [26]:
!pip install opencv-python
!pip install cmake
!pip install dlib
!pip install face-recognition
!pip install fer
!apt-get install ffmpeg


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 49 not upgraded.


## Importando Bibliotecas

In [27]:
import cv2
import dlib
import face_recognition
from fer import FER

### Inicializar o detector de emoções FER

In [28]:
emotion_detector = FER()

### Carregar o classificador de cascata do OpenCV

In [29]:
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

### Inicializar o detector de rostos Dlib

In [21]:
dlib_detector = dlib.get_frontal_face_detector()

### Carregar o vídeo

In [22]:
video_path = '/content/drive/MyDrive/Visao_computacional/Unlocking_Facial_Recognition_Diverse Activities_Analysis.mp4'
cap = cv2.VideoCapture(video_path)

### Obter propriedades do vídeo original para configurar o VideoWriter

In [30]:
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

### Configurar o VideoWriter para salvar o novo vídeo

In [35]:
output_path = '/content/drive/MyDrive/Visao_computacional/Video_Analisado.avi'
fourcc = cv2.VideoWriter_fourcc(*'XVID')  # Codec XVID para salvar temporariamente
out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

### Configurar o VideoWriter para salvar o novo vídeo

In [14]:
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detecção de rostos usando OpenCV
    opencv_faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    dlib_faces = dlib_detector(gray, 1)
    face_locations = face_recognition.face_locations(frame)

    all_faces = []
    for (x, y, w, h) in opencv_faces:
        all_faces.append((y, x + w, y + h, x))

    for face in dlib_faces:
        all_faces.append((face.top(), face.right(), face.bottom(), face.left()))

    all_faces.extend(face_locations)
    all_faces = list(set(all_faces))

    for (top, right, bottom, left) in all_faces:
        face_frame = frame[top:bottom, left:right]
        emotion, score = emotion_detector.top_emotion(face_frame) or ("Desconhecido", 0)
        cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)
        cv2.putText(frame, f'{emotion} ({score:.2f})', (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    out.write(frame)

cap.release()
out.release()
cv2.destroyAllWindows()

In [36]:
!ffmpeg -i /content/drive/MyDrive/Visao_computacional/Video_Analisado.avi -vcodec libx264 /content/drive/MyDrive/Visao_computacional/Video_Analisado.mp4

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab